# YOLO-pose 추출 + YOLO backend 학습

이 노트북은 다음 순서로 실행합니다.

1. 환경/경로 설정
2. `ultralytics` 설치 여부 확인
3. YOLO-pose COCO-17 keypoint 추출
4. 추출된 npz shape 확인
5. MediaPipe / YOLO overlay 시각화
6. `pose_backend=yolo` 로 phase 학습 실행

> 주의: YOLO 추출과 full 학습은 오래 걸릴 수 있습니다. `LIMIT`으로 먼저 smoke 실행 후 full로 바꾸는 것을 권장합니다.


In [1]:
from pathlib import Path
import os
import sys
import subprocess
import importlib
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'model' / 'train_ablation.py').exists():
    # 노트북을 다른 위치에서 열었을 때 repo root로 맞추려면 여기를 수정하세요.
    PROJECT_ROOT = Path(r'C:/Users/kojy1/PycharmProjects/capstone')
os.chdir(PROJECT_ROOT)

from model import train_ablation as ta

print('PROJECT_ROOT =', PROJECT_ROOT)
print('python       =', sys.executable)
print('YOLO dir     =', ta.YOLO_POSE_DIR)
print('video dir    =', ta.VIDEO_DIR)
print('label dir    =', ta.LABEL_DIR)


PROJECT_ROOT = C:\Users\kojy1\PycharmProjects\capstone
python       = C:\Users\kojy1\anaconda3\envs\capstone\python.exe
YOLO dir     = C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\STGCN_LSTM\poses_yolo
video dir    = C:\Users\kojy1\PycharmProjects\capstone\data\측면\data
label dir    = C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\labels


In [2]:
def run_stream(cmd, *, cwd=PROJECT_ROOT, env=None):
    """Run a long command and stream stdout/stderr live in Jupyter."""
    import os
    import subprocess
    import sys

    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    merged_env['PYTHONUNBUFFERED'] = '1'
    merged_env.setdefault('PYTHONIOENCODING', 'utf-8')

    print('RUN:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=merged_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    assert proc.stdout is not None
    while True:
        chunk = proc.stdout.read(1)
        if not chunk:
            break
        sys.stdout.write(chunk.decode('utf-8', errors='replace'))
        sys.stdout.flush()
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)
    return rc


## 1. 실행 옵션

- `LIMIT=None` 이면 전체 영상 추출
- `LIMIT=1` 또는 작은 숫자로 먼저 smoke test 가능
- `EXERCISE_TYPE=None` 이면 squat/benchpress/deadlift 전체


In [3]:
YOLO_MODEL = 'yolov8m-pose.pt'
DEVICE = 'cuda:0'     # 예: '0', 'cuda:0', 'cpu'. None이면 ultralytics 기본값
EXERCISE_TYPE = None   # 예: 'benchpress' 또는 None
LIMIT = None             # smoke: 1, full 추출: None
OVERWRITE = False
PHASE_HEAD_TYPE = 'mlp'  # 'mlp' or 'exercise_attn'

YOLO_OUTPUT_DIR = ta.YOLO_POSE_DIR
TRAIN_OUTPUT_ROOT = PROJECT_ROOT / 'phase_experiments' / 'yolo_pose' / 'full'

print('YOLO_MODEL       =', YOLO_MODEL)
print('DEVICE           =', DEVICE)
print('EXERCISE_TYPE    =', EXERCISE_TYPE)
print('LIMIT            =', LIMIT)
print('YOLO_OUTPUT_DIR  =', YOLO_OUTPUT_DIR)
print('PHASE_HEAD_TYPE =', PHASE_HEAD_TYPE)
print('TRAIN_OUTPUT_ROOT=', TRAIN_OUTPUT_ROOT)


YOLO_MODEL       = yolov8m-pose.pt
DEVICE           = cuda:0
EXERCISE_TYPE    = None
LIMIT            = None
YOLO_OUTPUT_DIR  = C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\STGCN_LSTM\poses_yolo
TRAIN_OUTPUT_ROOT= C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\yolo_pose\full


## 2. ultralytics 설치 확인

설치되어 있지 않으면 아래 셀의 `%pip install ultralytics` 주석을 해제하고 실행하세요.


In [4]:
# 필요 시 주석 해제
# %pip install ultralytics

try:
    import ultralytics
    print('ultralytics OK:', ultralytics.__version__)
except ImportError as e:
    print('ultralytics가 설치되어 있지 않습니다. 위의 %pip install ultralytics 를 실행하세요.')
    raise


ultralytics OK: 8.4.62


## 3. YOLO-pose 추출 실행

기본 출력 경로는 `data/.../STGCN_LSTM/poses_yolo/{type}/{video_stem}.npz` 입니다.


In [5]:
cmd = [
    sys.executable,
    '-u',
    'model/extract_pose_yolo.py',
    '--model', YOLO_MODEL,
    '--out-dir', str(YOLO_OUTPUT_DIR),
]
if DEVICE:
    cmd += ['--device', str(DEVICE)]
if EXERCISE_TYPE:
    cmd += ['--type', EXERCISE_TYPE]
if LIMIT is not None:
    cmd += ['--limit', str(LIMIT)]
if OVERWRITE:
    cmd += ['--overwrite']

run_stream(cmd, cwd=PROJECT_ROOT)


RUN: C:\Users\kojy1\anaconda3\envs\capstone\python.exe -u model/extract_pose_yolo.py --model yolov8m-pose.pt --out-dir C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\STGCN_LSTM\poses_yolo --device cuda:0
yolo-pose:   0%|          | 0/208 [00:00<?, ?it/s]olo-pose: 100%|������������������������������| 208/208 [00:00<00:00, 51944.94it/s]
YOLO extraction complete: success=208 failed=0 fail_log=n/a


0

## 4. YOLO npz 캐시 확인

추출된 파일 수와 sample shape `[T,17,3]` 를 확인합니다.


In [6]:
labels = ta.load_labels(ta.LABEL_DIR)
yolo_files = sorted(Path(YOLO_OUTPUT_DIR).glob('*/*.npz'))
print('YOLO npz count:', len(yolo_files))

if not yolo_files:
    raise FileNotFoundError('YOLO npz가 없습니다. 추출 셀을 먼저 실행하세요.')

sample_npz = yolo_files[0]
with np.load(sample_npz) as d:
    kpts = d['kpts']
    print('sample:', sample_npz)
    print('kpts shape:', kpts.shape)
    print('fps:', float(d['fps']) if 'fps' in d else None)
assert kpts.ndim == 3 and kpts.shape[1:] == (17, 3), kpts.shape
print('YOLO cache shape PASS')


YOLO npz count: 208
sample: C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\STGCN_LSTM\poses_yolo\benchpress\bench press_1.npz
kpts shape: (58, 17, 3)
fps: 29.97002997002997
YOLO cache shape PASS


## 5. MediaPipe / YOLO overlay 확인

같은 영상에 대해 MediaPipe npz와 YOLO npz가 모두 있을 때 overlay 이미지를 생성합니다.


In [7]:
from model.compare_pose_overlay import compare_pose_overlay

candidate = None
for (typ, name), _info in labels.items():
    mp_path = ta.pose_path(typ, name, pose_backend='mediapipe')
    yolo_path = ta.pose_path(typ, name, pose_backend='yolo', yolo_pose_dir=YOLO_OUTPUT_DIR)
    if mp_path.exists() and yolo_path.exists():
        candidate = (typ, name)
        break

if candidate is None:
    raise FileNotFoundError('MediaPipe와 YOLO npz가 모두 있는 영상을 찾지 못했습니다.')

typ, name = candidate
out = compare_pose_overlay(typ, name, yolo_pose_dir=YOLO_OUTPUT_DIR, frames=16)
print('overlay saved:', out)


overlay saved: C:\Users\kojy1\PycharmProjects\capstone\data\측면\data\STGCN_LSTM\pose_overlay\squat_squat_2_mediapipe_yolo.jpg


## 6. YOLO backend 학습 실행

`train_ablation.py`를 `PHASE_POSE_BACKEND=yolo` 로 실행합니다.

Smoke 학습을 먼저 하고 싶으면 `PHASE_EPOCHS=1`, `PHASE_MAX...` 같은 제한 변수를 추가로 넣어 사용하세요. 현재 셀은 full 기본 설정입니다.


In [8]:
env = os.environ.copy()
env.update({
    'PHASE_POSE_BACKEND': 'yolo',
    'PHASE_YOLO_POSE_DIR': str(YOLO_OUTPUT_DIR),
    'PHASE_ABLATION_ROOT': str(TRAIN_OUTPUT_ROOT),
    'PHASE_HEAD_TYPE': PHASE_HEAD_TYPE,
    'PHASE_EXTRACT_POSE': '0',   # YOLO 모드에서 MediaPipe fallback/extract 금지
    'PHASE_MODEL_TYPES': 'mlp',
    'PHASE_POOLINGS': 'temporal_avg',
    'PHASE_DERIVATIVE_MODES': 'pose',
    'PHASE_CLIP_LENS': '16',
    'PHASE_TRAIN_STRIDES': '2',
    'PHASE_AUGS': 'true',
})

cmd = [sys.executable, '-u', 'model/train_ablation.py']
print('output root:', TRAIN_OUTPUT_ROOT)
run_stream(cmd, cwd=PROJECT_ROOT, env=env)


output root: C:\Users\kojy1\PycharmProjects\capstone\phase_experiments\yolo_pose\full
RUN: C:\Users\kojy1\anaconda3\envs\capstone\python.exe -u model/train_ablation.py
pose extraction skipped
device: cuda | torch: 2.11.0+cu128 | numpy: 2.4.4
ROOT     : C:\Users\kojy1\PycharmProjects\capstone\data\������\data
WORK_DIR : C:\Users\kojy1\PycharmProjects\capstone\data\������\data\STGCN_LSTM
POSE     : yolo | V: 17 | graph: coco17
labels   : 210
type   benchpress  deadlift  squat
split                             
train          75        20     71
val            19         5     18
total experiments: 1
  [01] yolo_coco17_mlp_h128_c16_pw_2.0_ts2_do03_aug1

[EXP] yolo_coco17_mlp_h128_c16_pw_2.0_ts2_do03_aug1
  pose=yolo(coco17, V=17) model=mlp pooling=temporal_avg input=pose channels=3 hidden=128 clip=16 stride=2 drop=0.3 aug=True epochs=60 batch=64
  train=56322 | val=11700 windows
  params=1,839,020 | phase_head=41,347 | pooling_dim=256
  -> resume from epoch 9
ep09:   0%|          | 0/880 

KeyboardInterrupt: 

## 7. 결과 확인

학습 결과 CSV/JSON과 best checkpoint 위치를 확인합니다.


In [ ]:
result_csv = TRAIN_OUTPUT_ROOT / 'ablation_results.csv'
result_json = TRAIN_OUTPUT_ROOT / 'ablation_results.json'
print('result_csv :', result_csv, result_csv.exists())
print('result_json:', result_json, result_json.exists())

if result_csv.exists():
    import pandas as pd
    df = pd.read_csv(result_csv)
    display_cols = [c for c in [
        'exp_name', 'pose_backend', 'num_kpt', 'pose_graph_id',
        'phase_macro_f1', 'phase_acc', 'video_phase_f1',
        'best_epoch', 'checkpoint_path'
    ] if c in df.columns]
    display(df[display_cols].tail())
